## Threads in tkinter
Warum braucht man Threads in Tkinter?

Tkinter hat nur einen einzigen Haupt-Thread, der:

* die Oberfläche rendert

* Buttons anklickbar macht

* Events verarbeitet

Beispiel ohne Thread:

In [ ]:
import threading
import time
import tkinter as tk
from tkinter import ttk

def lange_aufgabe(label):
    """Diese Funktion läuft im Hintergrund-Thread"""
    for i in range(5):
        time.sleep(1)  # Lange Berechnung simulieren
        label.config(text=f"Fortschritt: {i+1}/5")


root = tk.Tk()
root.title("Tkinter + Threads Beispiel")

label = ttk.Label(root, text="Klicke auf Start...")
label.pack(pady=20)

button = ttk.Button(root, text="Start", command=lambda: lange_aufgabe(label))
button.pack(pady=10)

root.mainloop()


Das Ergebnis: Die ganze GUI friert ein.

### Lösung: Arbeit in einen Hintergrund-Thread auslagern
mit: <br><br>
<code>threading.Thread(target=lange_aufgabe).start()</code>

können Zeitaufwendige Aufgaben in einen anderen Thread ausgelagert werden:

In [ ]:
import threading
import time
import tkinter as tk
from tkinter import ttk

def lange_aufgabe(label):
    """Diese Funktion läuft im Hintergrund-Thread"""
    for i in range(5):
        time.sleep(1)  # Lange Berechnung simulieren
        label.config(text=f"Fortschritt: {i+1}/5")

def start_thread(label):
    """Startet die lange Aufgabe in einem eigenen Thread"""
    thread = threading.Thread(target=lange_aufgabe, args=(label,))
    thread.start()

root = tk.Tk()
root.title("Tkinter + Threads Beispiel")

label = ttk.Label(root, text="Klicke auf Start...")
label.pack(pady=20)

button = ttk.Button(root, text="Start", command=lambda: start_thread(label))
button.pack(pady=10)

root.mainloop()


#### Kurze, einfache Erklärung

Tkinter arbeitet nur in einem Haupt-Thread.

Eine lange Aufgabe würde dort die GUI blockieren → Fenster friert ein.

Deshalb starten wir einen Thread, der die lange Aufgabe erledigt.

In [ ]:
threading.Thread(target=lange_aufgabe).start()


Der Thread darf aber nicht direkt die GUI ändern.

Also schicken wir das Update zurück an Tkinter:

In [ ]:
label.after(0, ...)

Damit wird das Label wieder im GUI-Thread aktualisiert — sicher und korrekt.

⚠ Wichtig: Tkinter ist nicht Thread-Safe

Tkinter mag keine direkten Änderungen aus einem Worker-Thread.
Beim einfachen Beispiel oben geht es meistens gut, aber sauberer wäre:

Änderungen aus dem Thread per after() zurück in die GUI schicken

Die saubere Variante:

In [ ]:
def lange_aufgabe(label):
    for i in range(5):
        time.sleep(1)
        label.after(0, lambda i=i: label.config(text=f"Fortschritt: {i+1}/5"))


→ after(0, ...) führt die GUI-Änderung sicher im Tkinter-Thread aus.

### Das Beispiel mit einem weiten Objekt, den Fortschrittsbalken
✔ GUI friert nicht ein <br>
✔ Fortschrittsbalken wird sauber aktualisiert<br>
✔ Thread arbeitet im Hintergrund <br>
✔ Updates per after() zurück in die GUI <br>

In [ ]:
import threading
import time
import tkinter as tk
from tkinter import ttk

def lange_aufgabe(progress):
    """Lange Aufgabe in einem Hintergrund-Thread"""
    for i in range(101):  # 0 bis 100 %
        time.sleep(0.05)  # Rechenzeit simulieren

        progress.after(0, lambda i=i: progress.configure(value=i))

def start_thread(progress):
    """Startet den Hintergrund-Thread"""
    threading.Thread(target=lange_aufgabe, args=(progress,), daemon=True).start()

root = tk.Tk()
root.title("Fortschrittsbalken + Thread")

# Fortschrittsbalken
progress = ttk.Progressbar(root, orient="horizontal", length=300, mode="determinate", maximum=100)
progress.pack(pady=20)

# Button zum Starten
button = ttk.Button(root, text="Start", command=lambda: start_thread(progress))
button.pack(pady=10)

root.mainloop()

### Queue


Ein Worker-Thread schreibt Nachrichten in eine Queue.

Die GUI fragt die Queue ab und schreibt die Nachrichten ins Textfeld.

So siehst du direkt, wie man Threads und GUI sauber trennt.



In [ ]:
import threading
import time
import queue
import tkinter as tk
from tkinter import scrolledtext

def worker_task(q):
    """Worker-Thread schreibt Nachrichten in die Queue"""
    for i in range(1, 11):
        time.sleep(0.5)  # Arbeit simulieren
        q.put(f"Nachricht {i} vom Worker-Thread")
    q.put("Fertig!")  # Signal, dass alles erledigt ist

def start_thread(q):
    threading.Thread(target=worker_task, args=(q,), daemon=True).start()

def check_queue(q, text_widget):
    """Fragt die Queue ab und schreibt Nachrichten ins Textfeld"""
    try:
        while True:
            message = q.get_nowait()
            text_widget.insert(tk.END, message + "\n")
            text_widget.see(tk.END)  # Scrollt automatisch nach unten
    except queue.Empty:
        pass
    root.after(100, check_queue, q, text_widget)

root = tk.Tk()
root.title("Tkinter + Thread + Queue + Textfeld")

# Thread → GUI Kommunikation
q = queue.Queue()

# Textfeld für Ausgaben
text_area = scrolledtext.ScrolledText(root, width=40, height=10)
text_area.pack(padx=10, pady=10)

# Start-Button
button = tk.Button(root, text="Start Worker", command=lambda: start_thread(q))
button.pack(pady=5)

# Queue-Polling starten
root.after(100, check_queue, q, text_area)

root.mainloop()


Warum Queue? <br>

* Sie verhindert direkte GUI-Zugriffe aus Threads.

* Sie ist 100% thread-sicher.

* Sie funktioniert auch bei mehreren Events/Werten pro Tick.

* Sie ist das sauberste Muster für Tkinter-Threading.

<img style="float: center;" src="img/wbs-logo.jpg">


Autor: Dirk Maric
### Abbildungs- und Quellenverzeichnis
https://de.wikipedia.org/wiki/Python_(Programmiersprache)
Das Python Logo ist ein eingetragenes Warenzeichen der Python Software Foundation
Alle auf dieser Website veröffentlichten Logos sowie Marken-, Produkt- und Warenzeichen sind Eigentum der jeweiligen Unternehmen
© WBS TRAINING AG – Alle Rechte vorbehalten

### Nutzungsrechte:
Die Nutzung dieser Dokumentation ist ausschließlich für Schulungszwecke der WBS TRAINING AG gestattet. Eine Weitergabe an Dritte, auch auszugsweise, sowie Vervielfältigungen und Verbreitungen aller Art (elektronische und andere Verfahren) inklusive Übersetzungen sind nur mit vorheriger schriftlicher Zustimmung des Rechtinhabers gestattet. Zuwiderhandlungen verpflichten zu Schadenersatz.

### Herausgeber:

WBS TRAINING AG
Lorenzweg 5
12099 Berlin
Haftungsausschluss:
Alle Inhalte sind nach bestem Wissen korrekt und vollständig recherchiert und mit größtmöglicher Sorgfalt für die Schulungsunterlage zusammengestellt. Wir sind um die laufende Aktualisierung aller Informationen und Daten bemüht. Dennoch können Fehler (z.B. Abweichungen zur beschriebenen Hard- und Software durch kurzfristige Updates) auftreten, sodass wir für die vollständige Übereinstimmung, Richtigkeit und Aktualität keine Gewähr übernehmen. Hinweise unserer Nutzer werden konsequent weiterverfolgt.
